# plot_01d. ERA5: observed change (figures)

Drawn from what calc_03c saved, so nothing slow runs: the detection maps of calc_03c's section 3 (the section keeps its number), each model's class at every grid point and the number of models where ERA5's 1979–2013 trend is detected and consistent. calc_03c draws its signal-to-noise and record-rate figures itself. The model evaluation is plot_01a to plot_01c.

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import ListedColormap

from plotting_modules.constants import SEASONS
from extant import observed_change as oc
from extant.config import VARIABLES
from extant.utils.logger import setup_logging
from extant.plots import antarctica, forced_response as fr, observed_change as ocp
from extant.plots import response_change as rcp

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [2]:
%matplotlib inline

## Settings

In [ ]:
#(t): As in calc_03c: the variable (its name and units)
VARIABLE = VARIABLES['tas']

In [4]:
#(t): Where calc_03a to calc_03c saved their results: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

PosixPath('/work/scratch-pw4/aborow/ExtAnt')

## Which Results

Run **one** of the next two sections: each sets the folder the results are opened from, the sample's or the full data's.

### Sample data

In [5]:
#(t): The sample's results
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'sample'
RESULTS_DIR

PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/tas/sample')

### Full data

In [6]:
#(t): The full data's results
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE.name / 'full'
RESULTS_DIR

PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/tas/full')

## Open the Results

First what calc_03a to calc_03c saved in `RESULTS_DIR`, then calc_03c's detection maps opened.

In [7]:
#(t): What calc_03a to calc_03c saved in RESULTS_DIR
RESULTS_DIR, sorted(path.name for path in RESULTS_DIR.glob('*.zarr'))

(PosixPath('/work/scratch-pw4/aborow/ExtAnt/results/tas/full'),
 ['anomaly_ranks.zarr',
  'demo_anomaly_ranks.zarr',
  'demo_detrended_ranks.zarr',
  'demo_plume.zarr',
  'demo_statistics.zarr',
  'demo_tests.zarr',
  'detections.zarr',
  'detrended_ranks.zarr',
  'era5_kurtosis.zarr',
  'era5_lower_tail.zarr',
  'era5_mean.zarr',
  'era5_point.zarr',
  'era5_regional.zarr',
  'era5_regional_trend.zarr',
  'era5_skewness.zarr',
  'era5_std.zarr',
  'era5_trend.zarr',
  'era5_upper_tail.zarr',
  'era5_width.zarr',
  'eval_map_table.zarr',
  'eval_table.zarr',
  'experiment_noise.zarr',
  'final_mean.zarr',
  'final_quantiles.zarr',
  'first_look.zarr',
  'forced.zarr',
  'hist_nat_final_quantiles.zarr',
  'histnat_point.zarr',
  'historical_kurtosis.zarr',
  'historical_lower_tail.zarr',
  'historical_mean.zarr',
  'historical_point.zarr',
  'historical_skewness.zarr',
  'historical_std.zarr',
  'historical_trend.zarr',
  'historical_upper_tail.zarr',
  'historical_width.zarr',
  'mean_

In [ ]:
#(t): calc_03c's detection maps: each grid point's class and ERA5's percentile among each experiment's members, on
#(t): (model, season, lat, lon)
detections_ds = xr.open_zarr(RESULTS_DIR / 'detections.zarr').load()

## 3. Detection Maps

The trend test at every grid point: detected (outside hist-nat's 5–95%) and consistent (inside historical's). At p < 0.1 about 10% of points land outside a range by chance; read coherent regions.

### Calculations

In [ ]:
#(t): How many models call each point detected and consistent (class 1), on (season, lat, lon)
detection_counts_da = (detections_ds['detection_class'] == 1).sum('model')

In [ ]:
#(t): The number of models with a class anywhere: the most that can call a point detected and consistent (a model
#(t): without hist-nat, or with too few members to test, has none)
n_models_da = detections_ds['detection_class'].notnull().any(['season', 'lat', 'lon']).sum('model')
n_models_da

### Figures

In [ ]:
#(t): Is the observed (ERA5) trend detectable, and is historical consistent with it: every model (rows) and season, the
#(t): detection classes (oc.DETECTION_CLASSES); a map with no class at all is an ensemble too small to test
detection_class_da = detections_ds['detection_class'].sel(season=SEASONS)
models = detection_class_da['model'].values
class_names = [name.replace(', ', ',\n') for name in oc.DETECTION_CLASSES.values()]
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=len(models), n_cols=len(SEASONS), n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=len(models), n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(detection_class_da, 'model', 'season', axes=axes, cax=cax,
                                  levels=np.arange(-0.5, len(ocp.DETECTION_COLORS)), cmap=ListedColormap(ocp.DETECTION_COLORS),
                                  extend='neither', ticks=range(len(ocp.DETECTION_COLORS)), ticklabels=class_names,
                                  label="Detected: outside hist-nat's 5–95%. Consistent: inside historical's 5–95%.")
#(c): Say so where a map has no class at all, rather than leave it blank
for i, model in enumerate(models):
    for j, season in enumerate(SEASONS):
        if bool(detection_class_da.sel(model=model, season=season).isnull().all()):
            axes[i, j].text(0.5, 0.5, 'too few members\nto test (N < 20)', transform=axes[i, j].transAxes,
                            ha='center', va='center', fontsize=9)
antarctica.label_columns(axes, SEASONS)
antarctica.label_rows(axes, models)
fig.suptitle('Is the observed (ERA5) trend detectable, and is historical consistent with it?');

In [ ]:
#(t): How many models call each point detected and consistent (class 1), every season (white for none)
n_detection_models = int(n_models_da)
detection_count_cmap, _ = rcp.count_colormap(n_detection_models, 'Blues')
fig = plt.figure(figsize=antarctica.map_figure_size(n_rows=1, n_cols=len(SEASONS), n_colorbar_rows=1),
                 layout='constrained')
axes = antarctica.map_axes(fig, n_rows=1, n_cols=len(SEASONS))
cax = antarctica.colorbar_axis_below(axes)

antarctica.heatmaps_of_antarctica(detection_counts_da.sel(season=SEASONS), None, 'season', axes=axes, cax=cax,
                                  levels=np.arange(-0.5, n_detection_models + 1.5), cmap=detection_count_cmap,
                                  extend='neither', ticks=range(n_detection_models + 1), label='Number of models')
antarctica.label_columns(axes, SEASONS)
fig.suptitle(f'Number of models where the observed trend is {oc.DETECTION_CLASSES[1]}');